# 07 — Robustness to simulated haze (optional, no retraining)

**Purpose:** Probe whether the RGB/13-band advantage survives **synthetic atmospheric-looking perturbations**. For each available trained Random Forest, SmallCNN and ResNet18 checkpoint, evaluate the *unchanged held-out test set* under **uniform** or **wavelength-dependent** haze at fixed intensities; no model is retrained. Predictions and plots are derived from the real checkpoints and original GeoTIFF/cache data.

**Functions:** `src/piece_of_crop/haze.py` (`simulate_haze`, `HazyTestDataset`, `load_deep_model`, `predict_deep`, `predict_rf`). The models are rebuilt with `build_cnn` and `build_resnet`, the same functions that notebooks 04 and 05 trained with, so the saved weights always fit.

**Requires:** the complete preprocessing outputs from `02`, the test CSVs and local model files from `03`, `04`, `05`. Notebook `04` saves `models/cnn_<input>_seed<n>.pt`; notebook `03` saves `models/rf_<input>_seed<n>.joblib`; notebook `05` saves `models/resnet_<input>_seed<n>.pt`. Missing models are **skipped with a clear message**, never assigned invented performance. For final robustness of every model, use all three seeds.

**IMPORTANT:** The haze transform below is a **heuristic stress test**, not a radiative-transfer model nor physically calibrated Sentinel-2 simulation. No real cloud/haze labels are available in EuroSAT. Values mix raw pixel values with training-derived bright-reference band values. A positive or negative outcome must be described as *robustness to this synthetic perturbation only*, never real atmospheric performance.

**Run time:** Evaluation of many DL checkpoints can still take time, but does **not** retrain any model. Run after notebook 04/05 have finished. Start with `RUN_SEEDS=(0,)`; for a complete analysis change to `(0,1,2)`. Set `MAX_TEST_SAMPLES=256` for a quick diagnostic only; the final report should use `None` to evaluate the entire held-out test set.

## 1. Locate data, model weights and test predictions

All functions come from the installed `piece_of_crop` package. The cache is memory-mapped, not read wholesale into memory.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score

from piece_of_crop import (CLASS_NAMES, WAVELENGTH_FACTORS, find_project_root, get_cache, get_device,
                           load_deep_model, predict_deep, predict_rf, read_split, simulate_haze)

RUN_SEEDS = (0,)  # change to (0,1,2) for complete optional robustness analysis
STRENGTHS = (0.0, 0.15, 0.30)
PROFILES = ("uniform", "wavelength")
MAX_TEST_SAMPLES = None  # set 256 for quick diagnostics; None for final full test
BATCH_SIZE = 32
SAVE_PREDICTIONS = True

project_root = find_project_root()
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"
predictions_dir = results_dir / "predictions"
models_dir = project_root / "models"
for folder in (results_dir, figures_dir, predictions_dir, models_dir):
    folder.mkdir(parents=True, exist_ok=True)

split = read_split(results_dir / "split.csv")
band_stats = pd.read_csv(results_dir / "band_stats.csv")
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")
all_test = split.loc[split["split"].eq("test")].reset_index(drop=True)
test_rows = all_test.iloc[:MAX_TEST_SAMPLES] if MAX_TEST_SAMPLES is not None else all_test
if not len(test_rows):
    raise ValueError("Test set is empty.")
label_ids = np.arange(len(CLASS_NAMES), dtype=int)
assert cache.shape[0] == len(split)
device = get_device()

print("Project root:", project_root)
print("Device:", device, "| test patches:", len(test_rows), "/", len(all_test))

## 2. Define synthetic haze scenarios

For strength `a`, transform **raw** image bands: `x_hazy = (1 - a_band) * x + a_band * bright_reference_band`, where `bright_reference_band` is the **99.9th percentile clipping limit computed on training data** in notebook 02. `uniform`: same `a` for all bands; `wavelength`: hand-set multipliers favour larger perturbations in short visible wavelengths. These are transparent heuristics, not physically measured haze radiances. Normalization is applied **after** the perturbation using the original training statistics only.

In [ ]:
bright_reference = band_stats["clip_high"].to_numpy(dtype=np.float32)
assert bright_reference.shape == (13,) and np.isfinite(bright_reference).all()
print("Wavelength factors (B01 ... B8A):", WAVELENGTH_FACTORS)

example = np.asarray(cache[int(test_rows.iloc[0]["cache_row"])], dtype=np.float32)
assert np.array_equal(example, simulate_haze(example, 0.0, "uniform", bright_reference))
assert np.array_equal(example, simulate_haze(example, 0.0, "wavelength", bright_reference))
print("Haze=0 identity tests passed. Scenario strengths:", STRENGTHS)

## 3. Rebuild the saved models and predict on hazy patches

`load_deep_model` rebuilds the CNN with `build_cnn` (notebook 04) and the ResNet with `build_resnet` (notebook 05) and loads the saved weights with `strict=True`, so a mismatch stops with an error. `predict_deep` adds haze to the raw bands, normalises with the training statistics and predicts. `predict_rf` does the same with the random forest features from notebook 03. We verify that the **zero-haze predictions match the saved clean test CSV** before accepting any haze result.

## 4. Evaluate each available saved checkpoint, no retraining

A zero-haze sanity check **must reproduce clean predictions** for each model, input and seed; otherwise the model/haze result is not comparable. To save time while checking code, set `MAX_TEST_SAMPLES=256`. Use `None` and all three seeds for a full robustness table. Saved output names have `haze_` prefix so notebook 06 will not accidentally treat haze predictions as clean test predictions.

In [ ]:
family_prefixes={'RandomForest':'rf','SmallCNN':'cnn','ResNet18':'resnet'}
rows=[]
missing=[]
for model_name,prefix in family_prefixes.items():
    for input_type in ('rgb','13band'):
        for seed in RUN_SEEDS:
            extension='.joblib' if prefix=='rf' else '.pt'
            checkpoint_path=models_dir/f'{prefix}_{input_type}_seed{seed}{extension}'
            clean_file=predictions_dir/f'{prefix}_{input_type}_seed{seed}.csv'
            if not checkpoint_path.is_file() or not clean_file.is_file():
                missing.append((model_name,input_type,seed))
                print('Skipping missing weights or predictions:',model_name,input_type,seed)
                continue
            reference=pd.read_csv(clean_file)
            required={'file','y_true','y_pred'}
            if not required.issubset(reference.columns):
                raise ValueError(f'{clean_file.name} is missing {required-set(reference.columns)}')
            reference=test_rows[['file','label']].merge(
                reference[['file','y_true','y_pred']],on='file',validate='one_to_one',sort=False)
            if len(reference)!=len(test_rows) or not np.array_equal(reference['label'],reference['y_true']):
                raise ValueError(f'Clean predictions not aligned: {clean_file.name}')

            print(f'\n== {model_name}, {input_type}, seed {seed} ==',flush=True)
            if prefix=='rf':
                model=joblib.load(checkpoint_path)
                image_size=None
            else:
                model,image_size=load_deep_model(models_dir,prefix,input_type,seed,device)

            for profile in PROFILES:
                for strength in STRENGTHS:
                    if prefix=='rf':
                        prob=predict_rf(model,test_rows,cache,band_stats,input_type,profile,strength,BATCH_SIZE)
                    else:
                        prob=predict_deep(model,image_size,test_rows,cache,band_stats,input_type,
                                          profile,strength,device,BATCH_SIZE)
                    pred=prob.argmax(axis=1)
                    if strength==0.:
                        matches=np.mean(pred==reference['y_pred'].to_numpy(dtype=int))
                        if matches!=1.:
                            raise ValueError(f'Zero-haze predictions do not reproduce clean model '
                                             f'{model_name}/{input_type}/seed{seed} '
                                             f'(agreement {matches:.3f}). Stop and check checkpoint/preprocessing.')
                    y_true=test_rows['label'].to_numpy(dtype=int)
                    row={'model':model_name,'input':input_type,'seed':seed,
                         'profile':profile,'strength':strength,'n_test':len(test_rows),
                         'accuracy':accuracy_score(y_true,pred),
                         'macro_f1':f1_score(y_true,pred,labels=label_ids,
                                             average='macro',zero_division=0),
                         'subset':len(test_rows)!=len(all_test)}
                    rows.append(row)
                    print(f'{profile} / a={strength:.2f} | acc={row["accuracy"]:.4f} '
                          f'F1={row["macro_f1"]:.4f}',flush=True)
                    if SAVE_PREDICTIONS:
                        out=pd.DataFrame({'file':test_rows['file'].to_numpy(),
                                          'y_true':y_true,'y_pred':pred,
                                          'confidence':prob.max(axis=1)})
                        out.to_csv(predictions_dir/
                                   f'haze_{prefix}_{input_type}_seed{seed}_{profile}_a{int(round(strength*100)):02d}.csv',
                                   index=False)
            del model
            if device.type=='mps' and hasattr(torch,'mps'):
                torch.mps.empty_cache()

if not rows:
    raise FileNotFoundError('No matching clean prediction CSV + model checkpoints. Run 03/04/05 first.')
haze_results=pd.DataFrame(rows)
# Delta is relative to the corresponding unperturbed scenario for that seed and profile.
base=haze_results.loc[haze_results['strength']==0,
                      ['model','input','seed','profile','accuracy','macro_f1']]
base=base.rename(columns={'accuracy':'clean_accuracy','macro_f1':'clean_macro_f1'})
haze_results=haze_results.merge(base,on=['model','input','seed','profile'],validate='many_to_one')
haze_results['accuracy_drop']=haze_results['clean_accuracy']-haze_results['accuracy']
haze_results['macro_f1_drop']=haze_results['clean_macro_f1']-haze_results['macro_f1']
output_path=results_dir/'haze_results.csv'
haze_results.to_csv(output_path,index=False)
print('Saved',output_path, 'Unavailable models:', missing)
display(haze_results.round(4))

## 5. Plot robustness curves and interpret cautiously

Plot **accuracy** and **macro-F1** against haze strength. If 13-band models degrade less than RGB, the evidence is only for the **defined synthetic transform** and should be evaluated across seeds. The default `(0,)` shows a pilot, not a three-seed final study. `07` remains optional according to the original project plan.

In [ ]:
import matplotlib.pyplot as plt
for model_name in haze_results['model'].unique():
    for profile in PROFILES:
        group=haze_results.query('model == @model_name and profile == @profile')
        if group.empty:
            continue
        fig,axes=plt.subplots(1,2,figsize=(11,4))
        for input_type in ('rgb','13band'):
            subset=group.query('input == @input_type')
            if subset.empty:
                continue
            mean=subset.groupby('strength')[['accuracy','macro_f1']].mean()
            axes[0].plot(mean.index,mean['accuracy'],marker='o',label=input_type)
            axes[1].plot(mean.index,mean['macro_f1'],marker='o',label=input_type)
        for ax,metric in zip(axes,('Accuracy','Macro-F1')):
            ax.set(title=f'{model_name}: {profile} haze | {metric}',
                   xlabel='Synthetic haze strength (unitless)',ylabel=metric)
            ax.grid(alpha=.25);ax.legend()
        fig.tight_layout()
        fig.savefig(figures_dir/f'haze_{model_name}_{profile}.png',dpi=150,bbox_inches='tight')
        plt.show()
print('Haze robustness files:',results_dir/'haze_results.csv')
print('Ran seeds:',RUN_SEEDS, 'Subset?', MAX_TEST_SAMPLES is not None)
print('CAVEAT: Synthetic stress test only. No real atmospheric haze claims.')

## 6. Optional-module conclusions and responsible reporting

- This analysis is an optional extension. The clean-model evaluation in `06` is the main evidence for the project question.
- Haze simulation uses linear mixing and manually assigned spectral factors; it is **not calibrated from real haze observations, optical depth, or atmospheric correction physics**.
- If checkpoint weights are missing, the notebook **skips** those models and states which are missing. Do not interpret a partial comparison as complete.
- The zero-strength replication check protects against inconsistent normalization, wrong checkpoint and wrong label alignment. **Do not remove that check**.
- For the final report, run on all held-out test patches with three seeds, report means and uncertainty, and describe the absence of real hazy EuroSAT labels.

**External methodological context:** [Remote Sensing, 2020, *Robust Single-Image Haze Removal Using Optimal Transmission Map and Adaptive Atmospheric Light*](https://www.mdpi.com/2072-4292/12/14/2233) discusses atmospheric scattering as an attenuation-plus-airlight model. This project only borrows its algebraic *form*; its particular band coefficients, bright reference and strengths are intentionally heuristic and **are not calibrated against the cited physical model**.